# Week 8 (Tue) - Class Activities in CS 315 (DuckDB Version)

Solving tasks from `netflix_data.db` using DuckDB SQL queries.

**Table of Contents**

0. [Q0: Connect to Database & List Tables](#q0)
1. [Q1: Inspect Table Data](#q1)
2. [Q2: Show Number of Rows](#q2)
3. [Q3: Show Column Names](#q3)
4. [Q4: Show Number of Total Countries](#q4)
5. [Q5: Show Number of TV Series](#q5)
6. [Q6: Number of TV Series in South Korea](#q6)
7. [Q7: Total Weeks of Squid Game in Italy](#q7)
8. [Q8: Time Frame for a Given Country](#q8)
9. [Q9: Is Number of Weeks in Ranking Equal?](#q9)
10. [Q10: Country with Fewest Weeks](#q10)

<a id="q0"></a>
### Q0: Connect to Database & List Tables

Connect to `netflix_data.db` and inspect the available tables in the database.

In [1]:
import duckdb

con = duckdb.connect('netflix_data.db')
con

When we open a connection it will remain open until we explicetely close it.

We can execute simple SQL commands as shown below.

In [2]:
con.sql("SHOW TABLES")

┌──────────┐
│   name   │
│ varchar  │
├──────────┤
│ films    │
│ tvseries │
└──────────┘

<a id="q1"></a>
### Q1: Load & Preview Data

Preview the top rows `tvseries`. This is similar to the `.head` method in pandas.

In [3]:
con.sql("SELECT * FROM tvseries LIMIT 5")

┌──────────────┬──────────────┬────────────┬─────────────┬──────────────────────┬───────────────────────────────────┬────────────────────────────┐
│ country_name │ country_iso2 │    week    │ weekly_rank │      show_title      │           season_title            │ cumulative_weeks_in_top_10 │
│   varchar    │   varchar    │  varchar   │    int64    │       varchar        │              varchar              │           int64            │
├──────────────┼──────────────┼────────────┼─────────────┼──────────────────────┼───────────────────────────────────┼────────────────────────────┤
│ Argentina    │ AR           │ 2026-10-04 │           1 │ Not a Stranger       │ Not a Stranger: Season 1          │                          3 │
│ Argentina    │ AR           │ 2026-10-04 │           2 │ My Sad Dead          │ My Sad Dead: Limited Series       │                          2 │
│ Argentina    │ AR           │ 2026-10-04 │           3 │ The Final Problem    │ The Final Problem: Limited Series │ 

<a id="q2"></a>
### Q2: Show number of rows

In [4]:
con.sql("SELECT COUNT(*) FROM tvseries")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       256100 │
└──────────────┘

Because the result is a table, if we want just the number, we will use the method `fetchone` to get the first row:

In [5]:
con.sql("SELECT COUNT(*) FROM tvseries").fetchone()

(256100,)

Since the result is a tuple, then we can index it like this:

In [6]:
con.sql("SELECT COUNT(*) FROM tvseries").fetchone()[0]

256100

<a id="q3"></a>
### Q3: Show column names

In [7]:
con.sql("DESCRIBE tvseries")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ country_name               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_iso2               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ week                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ weekly_rank                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ show_title                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ season_title               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cumulative_weeks_in_top_10 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────────┴─────────────┴─────────┴─────────┴─

<a id="q4"></a>
### Q4: Show number of total countries

In [8]:
con.sql("SELECT COUNT(DISTINCT country_name) FROM tvseries").fetchone()[0]

94

<a id="q5"></a>
### Q5: Show number of TV series

In [9]:
con.sql("SELECT COUNT(DISTINCT show_title) FROM tvseries").fetchone()[0]

3574

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [ ]:
con.sql("""
    SELECT COUNT(DISTINCT show_title) 
    FROM tvseries 
    WHERE country_name = 'South Korea'
""").fetchone()[0]


481

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [11]:
con.sql("""
    SELECT COUNT(DISTINCT week) 
    FROM tvseries 
    WHERE country_name = 'Italy' AND show_title = 'Squid Game'
""").fetchone()[0]

29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [12]:
def show_timeframe(con, country):
    """A function to print out the time frame a country was in the Netflix rankings using DuckDB."""
    res = con.sql("""
        SELECT MIN(week), MAX(week)
        FROM (
            SELECT country_name, week FROM tvseries
        )
        WHERE country_name = $1
    """, params=[country]).fetchone()
    
    start, end = res[0], res[1]
    print(f"Netflix rankings for {country} exist between {start} and {end}.")

show_timeframe(con, "Chile")
show_timeframe(con, "Russia")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-04.
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

Check if total rows match `20 * unique_weeks * unique_countries`.

In [13]:
is_equal = con.sql("""
    SELECT COUNT(*) = 10 * COUNT(DISTINCT week) * COUNT(DISTINCT country_name) 
    FROM tvseries
""").fetchone()[0]

is_equal

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Find the country with the fewest weeks in the Netflix rankings.

In [14]:
con.sql("""
SELECT country_name, COUNT(DISTINCT week) AS weekCnt
FROM tvseries
GROUP BY country_name
ORDER BY weekCnt ASC
LIMIT 5;""")

┌────────────────────┬─────────┐
│    country_name    │ weekCnt │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ Russia             │      35 │
│ Dominican Republic │     275 │
│ India              │     275 │
│ Germany            │     275 │
│ Malaysia           │     275 │
└────────────────────┴─────────┘

# TASK 3C

In [16]:
import duckdb

con = duckdb.connect('simple_netflix.db')
con

con.sql("SHOW TABLES")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ simple          │
│ sqlite_sequence │
└─────────────────┘

We see that there is a table called simple, what is sqlite_sequence? --> I googled this, and it's an internal system used to keep track of the maximum row identifier.

Let's show a preview of the table:

In [17]:
con.sql("SELECT * FROM simple LIMIT 5")

┌───────┬─────────┬────────────┬───────┬──────────┬──────────────────────────────────────────────────┐
│  id   │ country │    week    │ rank  │ category │                    show_title                    │
│ int64 │ varchar │  varchar   │ int64 │ varchar  │                     varchar                      │
├───────┼─────────┼────────────┼───────┼──────────┼──────────────────────────────────────────────────┤
│     1 │ AR      │ 2026-10-04 │     1 │ Films    │ UNABOMBER                                        │
│     2 │ AR      │ 2026-10-04 │     2 │ Films    │ The Equalizer 3                                  │
│     3 │ AR      │ 2026-10-04 │     3 │ Films    │ The Substance                                    │
│     4 │ AR      │ 2026-10-04 │     4 │ Films    │ Demon Slayer: Kimetsu no Yaiba Infinity Castle I │
│     5 │ AR      │ 2026-10-04 │     5 │ Films    │ What a Father!                                   │
└───────┴─────────┴────────────┴───────┴──────────┴──────────────────────

Let's describe the columns:

In [19]:
con.sql("DESCRIBE simple")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ PRI     │ NULL    │ NULL    │
│ country     │ VARCHAR     │ NO      │ NULL    │ NULL    │ NULL    │
│ week        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rank        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ show_title  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

What are all the unique "category" values?  I only know Film and TV - is that it?

In [20]:
con.sql("""
    SELECT DISTINCT category
    FROM simple 
""")

┌──────────┐
│ category │
│ varchar  │
├──────────┤
│ TV       │
│ Films    │
└──────────┘

Turns out, there are only TV and Films.

Let's find al the weeks with top 1 TV shows in the US.

In [21]:
con.sql("""
    SELECT week, show_title
    FROM simple 
    WHERE country = 'US' AND rank == 1
""")

┌────────────┬──────────────────────────────────┐
│    week    │            show_title            │
│  varchar   │             varchar              │
├────────────┼──────────────────────────────────┤
│ 2026-10-04 │ UNABOMBER                        │
│ 2026-10-04 │ A Different World                │
│ 2026-09-27 │ UNABOMBER                        │
│ 2026-09-27 │ Monster: The Lizzie Borden Story │
│ 2026-09-20 │ Why Did I Get Married Again?     │
│ 2026-09-20 │ Monster: The Lizzie Borden Story │
│ 2026-09-13 │ Why Did I Get Married Again?     │
│ 2026-09-13 │ Death of the Pastor's Wife       │
│ 2026-09-06 │ The Whisper Man                  │
│ 2026-09-06 │ Death of the Pastor's Wife       │
│     ·      │       ·                          │
│     ·      │       ·                          │
│     ·      │       ·                          │
│ 2021-08-01 │ Blood Red Sky                    │
│ 2021-08-01 │ Outer Banks                      │
│ 2021-07-25 │ Twilight                         │


Why are there two values for each week?    

Oh it's because I didn't choose category == TV

In [23]:
con.sql("""
    SELECT week, show_title
    FROM simple 
    WHERE category == 'TV' AND country == 'US' AND rank == 1
""")

┌────────────┬──────────────────────────────────────┐
│    week    │              show_title              │
│  varchar   │               varchar                │
├────────────┼──────────────────────────────────────┤
│ 2026-10-04 │ A Different World                    │
│ 2026-09-27 │ Monster: The Lizzie Borden Story     │
│ 2026-09-20 │ Monster: The Lizzie Borden Story     │
│ 2026-09-13 │ Death of the Pastor's Wife           │
│ 2026-09-06 │ Death of the Pastor's Wife           │
│ 2026-08-30 │ Death of the Pastor's Wife           │
│ 2026-08-23 │ Outer Banks                          │
│ 2026-08-16 │ My Life With the Walter Boys         │
│ 2026-08-09 │ The Idaho Murders: College Nightmare │
│ 2026-08-02 │ The Idaho Murders: College Nightmare │
│     ·      │     ·                                │
│     ·      │     ·                                │
│     ·      │     ·                                │
│ 2021-09-05 │ Clickbait                            │
│ 2021-08-29 │ Clickbait    

Let's further order this by descending order.

In [24]:
con.sql("""
    SELECT week, show_title
    FROM simple 
    WHERE category == 'TV' AND country == 'US' AND rank == 1
    ORDER BY week
""")

┌────────────┬──────────────────────────────────────┐
│    week    │              show_title              │
│  varchar   │               varchar                │
├────────────┼──────────────────────────────────────┤
│ 2021-07-04 │ Manifest                             │
│ 2021-07-11 │ Manifest                             │
│ 2021-07-18 │ Virgin River                         │
│ 2021-07-25 │ Virgin River                         │
│ 2021-08-01 │ Outer Banks                          │
│ 2021-08-08 │ Outer Banks                          │
│ 2021-08-15 │ Outer Banks                          │
│ 2021-08-22 │ All American                         │
│ 2021-08-29 │ Clickbait                            │
│ 2021-09-05 │ Clickbait                            │
│     ·      │     ·                                │
│     ·      │     ·                                │
│     ·      │     ·                                │
│ 2026-08-02 │ The Idaho Murders: College Nightmare │
│ 2026-08-09 │ The Idaho Mur

Great, now let's try to do groupby:

In [26]:
con.sql("""
SELECT country, COUNT(DISTINCT show_title) AS number_shows
FROM simple
WHERE category=='TV' AND rank ==1
GROUP BY country
ORDER BY number_shows ASC
""")

┌─────────┬──────────────┐
│ country │ number_shows │
│ varchar │    int64     │
├─────────┼──────────────┤
│ RU      │           12 │
│ IN      │           86 │
│ EG      │           95 │
│ VN      │           98 │
│ BG      │           98 │
│ BD      │           98 │
│ JP      │           99 │
│ PK      │          103 │
│ LV      │          104 │
│ LT      │          104 │
│ ·       │           ·  │
│ ·       │           ·  │
│ ·       │           ·  │
│ JM      │          136 │
│ IS      │          136 │
│ MQ      │          140 │
│ GP      │          140 │
│ BR      │          141 │
│ IE      │          142 │
│ RE      │          144 │
│ NC      │          146 │
│ RO      │          146 │
│ BS      │          146 │
├─────────┴──────────────┤
│ 94 rows      2 columns │
│ (20 shown)             │
└────────────────────────┘

It's important for the filtering "where" to come before the group by!